# Structured outputs with Pydantic

## Learning goals

- Turn model-generated JSON into a typed application boundary
- Define enums, ranges, required fields, and extra-field policy
- Distinguish JSON syntax, schema validity, and semantic correctness
- Implement bounded recovery for malformed output
- Use an SDK-native parsed response without making live calls by default


## Why structured output matters

Natural-language output is ideal for people but brittle for code. If an application must route a ticket, schedule a task, or choose a tool, it needs a stable contract. Pydantic models make that contract executable: they parse input, enforce field rules, reject unexpected shapes, and produce typed Python values.

Structure reduces one class of uncertainty; it does not make model claims true. A perfectly valid object can still contain the wrong route, fabricated facts, or an overconfident score. Validation must be paired with evaluation and domain checks.


## Mental model

```mermaid
flowchart LR
  A[Prompt plus schema] --> B[Model]
  B --> C[Candidate structured output]
  C --> D{Valid JSON?}
  D -->|no| E[Syntax failure]
  D -->|yes| F{Matches Pydantic schema?}
  F -->|no| G[Validation failure]
  F -->|yes| H[Typed Python object]
  H --> I{Semantically acceptable?}
  I -->|yes| J[Application action]
  I -->|no| K[Review, fallback, or refusal]
```

There are three separate gates: JSON parsing, schema validation, and business or semantic validation. Keep them separate in logs and metrics because each failure needs a different fix.


## Define the smallest useful schema

Use an enum when the application supports a closed set of actions. Constrain numbers and strings. Forbid unexpected fields when silently accepting them could hide a misspelling or unsupported behavior.


In [ ]:
from enum import StrEnum

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class RouteCategory(StrEnum):
    BILLING = "billing"
    ACCOUNT = "account"
    TECHNICAL = "technical"
    MULTI_INTENT = "multi_intent"


class TicketRoute(BaseModel):
    model_config = ConfigDict(extra="forbid")

    category: RouteCategory
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str = Field(min_length=5, max_length=240)
    requires_human_review: bool = False


print(TicketRoute.model_json_schema())


## Parse a valid JSON response

`model_validate_json` performs both JSON decoding and model validation. The result is a `TicketRoute`, so downstream code receives a `RouteCategory` enum and a bounded float rather than arbitrary dictionary values.


In [ ]:
raw_valid = """
{
  "category": "billing",
  "confidence": 0.92,
  "reason": "The customer reports a duplicate charge.",
  "requires_human_review": false
}
"""

route = TicketRoute.model_validate_json(raw_valid)
print(route)
print("Enum value:", route.category.value)
print("Serialized:", route.model_dump(mode="json"))


## Read validation errors as data

Pydantic reports the field location, error type, message, and rejected input. Convert this into safe diagnostic information; do not dump sensitive model input into production logs.


In [ ]:
invalid_candidates = {
    "invalid_json": '{"category": "billing",',
    "unsupported_enum": '{"category": "sales", "confidence": 0.7, "reason": "Asks for a quote"}',
    "out_of_range": '{"category": "billing", "confidence": 1.4, "reason": "Duplicate charge"}',
    "extra_field": '{"category": "billing", "confidence": 0.8, "reason": "Duplicate charge", "priority": "urgent"}',
}

for name, raw in invalid_candidates.items():
    try:
        TicketRoute.model_validate_json(raw)
    except ValidationError as error:
        first = error.errors(include_url=False)[0]
        print(name, {"field": first["loc"], "type": first["type"], "message": first["msg"]})


## Retry only with a reason and a cap

A retry can help when output is truncated or malformed. It does not solve an unsupported request, a policy refusal, or a consistently wrong schema. Always cap retries and preserve the final error for observability. This offline function simulates successive model candidates.


In [ ]:
def parse_with_bounded_recovery(
    candidates: list[str],
    *,
    max_attempts: int = 2,
) -> TicketRoute:
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")

    last_error: ValidationError | None = None
    for attempt, raw in enumerate(candidates[:max_attempts], start=1):
        try:
            parsed = TicketRoute.model_validate_json(raw)
            print(f"Accepted attempt {attempt}")
            return parsed
        except ValidationError as error:
            last_error = error
            print(f"Rejected attempt {attempt}: {error.error_count()} validation error(s)")

    raise ValueError("No valid structured output within the attempt limit") from last_error


recovered = parse_with_bounded_recovery([
    invalid_candidates["out_of_range"],
    raw_valid,
])
print(recovered.category)


## Schema-valid is not decision-valid

A model-provided confidence score is not automatically a calibrated probability. Define application policy separately. For example, route low-confidence or multi-intent cases to review regardless of whether the object is schema-valid.


In [ ]:
def routing_decision(route: TicketRoute, threshold: float = 0.80) -> str:
    if route.requires_human_review:
        return "human_review"
    if route.category is RouteCategory.MULTI_INTENT:
        return "human_review"
    if route.confidence < threshold:
        return "human_review"
    return route.category.value


print("Application decision:", routing_decision(route))


## Optional OpenAI parsed response

The OpenAI Python SDK can accept a Pydantic model as the response format and return `output_parsed`. This is preferable to asking for JSON in prose and manually scraping text. The live call remains disabled by default.


In [ ]:
import os

from openai import OpenAI

RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    client = OpenAI()
    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
        input=[
            {"role": "system", "content": "Route one support ticket."},
            {"role": "user", "content": "I was charged twice."},
        ],
        text_format=TicketRoute,
        store=False,
    )
    parsed_route = response.output_parsed
    print(parsed_route)
else:
    print("Live structured-output call skipped.")


## Failure cases and improvements

| Failure | Improvement |
|---|---|
| JSON wrapped in Markdown fences | Use native structured output or strip only known wrappers before validation |
| New fields silently ignored | Use `extra="forbid"` where strict contracts matter |
| Endless “please fix JSON” loop | Cap retries and surface the last validation error |
| Valid schema triggers wrong action | Add semantic rules and evaluation cases |
| Model confidence treated as truth | Calibrate against labeled data or use it only as a review signal |
| Schema changes break consumers | Version contracts and test backward compatibility |


## Exercise

1. Add an optional list of `sub_intents` that is required only for `multi_intent`.
2. Add a model validator enforcing that relationship.
3. Write tests for invalid JSON, missing fields, extra fields, and invalid enums.
4. Add a retry metric without logging the full ticket text.
5. Explain why a schema-valid high-confidence route may still require human review.


## Summary

Structured output turns model text into a typed boundary. Define narrow schemas, reject unsupported values, inspect validation errors, and cap recovery attempts. Then apply semantic and authorization rules before acting. Pydantic proves that data matches your declared shape; it does not prove that the model's decision is correct.
